# Week 2, Day 2: Python Foundations for Data Engineering
## Complete Self-Learning Notebook

This notebook unifies every drill, activity, and advanced exercise from Day 2 into one self-contained, runnable reference. Work through it top to bottom or jump to any section you want to revisit.

**How to use it**
1. Read the explanation cell before running any code.
2. Run the code cell and observe the output.
3. Read the output-explained cell that follows.
4. Experiment: change a value, predict what changes, then run again.

**Before you run it**: copy this notebook into your `student-work/week2/day2/` project (see `Activity_0_UV_Project_Setup.md`) and open it from there, selecting that project's `.venv` as the kernel. Several cells in sections 11 to 13 and the activities write small output files (`claims_summary.txt`, `policy_summary.csv`, `claims_totals.json`, `claims_intake_summary.json`) to the current folder. Running the notebook from `student-work/` keeps those files out of the instructor-provided `Week 2/Labs/Day 2/` folder, so a later `git pull` never conflicts with them.

**AI-Free Zone (Weeks 1-4):** Write all code yourself. The goal is muscle memory, not quick answers. You will move faster in Weeks 5-8 because you built the foundation now.

---
### Table of Contents
| # | Topic |
|---|-------|
| 01 | Variables and Data Types |
| 02 | Conditionals |
| 03 | Triage Faceoff (Interactive) |
| 04 | Loops |
| 05 | Lists |
| 06 | Dictionaries |
| 07 | Iterating Lists |
| 08 | Functions |
| 09 | Nested Data Structures |
| 10 | Classes |
| 11 | File I/O |
| 12 | CSV I/O |
| 13 | JSON I/O |
| 14 | Lambdas, map, and filter |
| 15 | SQLite |
| 16 | Error Handling |
| 17 | The collections Module |
| 18 | datetime |
| A1 | Advanced: Comprehensions |
| A2 | Advanced: Pattern Matching |
| A3 | Advanced: Dataclasses |
| A4 | Advanced: itertools |
| A5 | Advanced: Decorators |
|    | Mini Project 1: Claims Intake Pipeline |
|    | Mini Project 2: Appointment No-Show Analysis |

---
## 01: Variables and Data Types

### What it is
A **variable** is a named box that holds a value. Python infers the type from the value you assign, so you never write `double reserve = 5000.0;` with an explicit type like in Java or C.

The types you use most in data engineering:
| Type | Example | Notes |
|------|---------|-------|
| `int` | `42` | whole numbers |
| `float` | `5000.00` | decimals, money |
| `str` | `"CLM-1001"` | text, IDs |
| `bool` | `True` | flags |

### f-strings
`f"text {variable}"` is the modern way to format output (Python 3.6+). Inside the braces you can apply format specs: `{value:,.2f}` means comma-separated thousands, two decimal places.

### Why data engineers care
Every field in a CSV, JSON, or database row arrives as a value with a type. Knowing Python's types and how to format them cleanly is the foundation for every data-cleaning step you will do later.

### In the insurance domain
A claim reserve is the amount an insurer sets aside to pay a claim. When the claim settles, the settlement often differs from the reserve. The variance and percent change are key metrics for an actuary or claims manager.

In [ ]:
# Drill 01: Claim Reserve Variance

# Formulas
# Variance = Settlement - Reserve
# Percent Change = Variance / Reserve * 100

reserve = 5000.00      # float: dollars set aside
settlement = 6200.00   # float: actual payout

variance = settlement - reserve
percent_change = (variance / reserve) * 100

# Basic print
print(f"Claim CLM-1001 reserve was ${reserve}")
print(f"Claim CLM-1001 settled at ${settlement}")
print(f"Claim CLM-1001 reserve changed by {percent_change}%")

print()

# Challenge: currency formatting with thousands separator
print(f"Reserve:    ${reserve:,.2f}")
print(f"Settlement: ${settlement:,.2f}")
print(f"Variance:   ${variance:,.2f}")
print(f"Change:     {percent_change:.2f}%")

**Output explained**

The first three lines show the plain values. The second block uses format specs:
- `:,.2f` adds comma-separated thousands and exactly two decimal places (standard for money).
- `:.2f` gives exactly two decimal places with no thousands separator.

A positive variance means the claim cost more than reserved. A positive percent change tells the adjuster by how much the reserve underestimated the final payout. In this case the reserve was 24 percent too low.

---
## 02: Conditionals

### What it is
An `if` / `elif` / `else` block lets your code make a decision. Python evaluates the condition (a boolean expression) and runs the first branch that is `True`.

**Key operators**

| Operator | Meaning |
|----------|---------|
| `==` | equal |
| `!=` | not equal |
| `<` `<=` `>` `>=` | comparison |
| `and` `or` `not` | logical connectives |
| `in` `not in` | membership |
| `len(x)` | length of a string, list, etc. |

### Why data engineers care
Routing logic is everywhere: route a record to a queue, flag a value as invalid, decide which bucket a number falls into. Conditionals are the building block for all rule-based classification you will do before you learn ML-based classification later.

### In the insurance domain
Claims are routed based on amount, policy type, risk flags, and authority level. The five examples below walk through progressively complex routing decisions.

In [ ]:
# Drill 02: Conditionals

# 1. Simple comparison
claim_amount = 5000
if 2 * claim_amount > 10000:
    print("1: Escalate to senior adjuster")
else:
    print("1: Handle in standard queue")

# 2. String length check
policy = "AUTO"
if len(policy) < 5:
    print("2: Short policy code accepted")
else:
    print("2: Policy code too long")

# 3. Age-based routing
claim_age_days = 21
if claim_age_days > 20:
    print("3: Claim is overdue for review")
else:
    print("3: Claim is still within the review window")

# 4. Combined condition with exponent and logical and
severity = 2
flags = 5
if (severity ** 3 >= flags) and (flags ** 2 < 26):
    print("4: Route to SIU")
else:
    print("4: Standard handling")

# 5. Multi-branch authority check
payout = 18000
authority_level = 1
manager_override = True

if (payout < 2000) and (authority_level >= 1):
    print("5: Adjuster can approve directly")
elif (payout < 10000) and (authority_level >= 2):
    print("5: Needs a senior adjuster")
elif (payout < 25000) and (authority_level >= 3):
    print("5: Needs a claims manager")
elif ((payout < 50000) and (authority_level >= 3)) or (manager_override and (payout < 50000)):
    print("5: Approved via manager override")
else:
    print("5: Escalate to claims committee")

**Output explained**

1. `2 * 5000 = 10000`, which is NOT `> 10000` (strict), so we go to the `else` branch: standard queue.
2. `len("AUTO")` is 4, which is `< 5`, so the short code is accepted.
3. `21 > 20` is `True`, so the claim is overdue.
4. `2**3 = 8 >= 5` is `True`, and `5**2 = 25 < 26` is `True`, so both conditions hold: route to SIU.
5. None of the first three `elif` branches match (`payout = 18000` fails the `< 10000` threshold and `authority_level = 1` fails the `>= 3` threshold), but the fourth branch matches because `manager_override = True` and `payout < 50000`. Result: manager override approved.

**Key takeaway**: conditions are evaluated top to bottom. Python stops at the first `True` branch. Order your most-specific conditions first.

---
## 03: Triage Faceoff (Interactive)

### What it is
This is the same logic as the conditionals drill, but wrapped in a small game. The system assigns a random severity; you guess it. This is the classic pattern for any assessment tool: compare two values and report whether they match, and in which direction they differ.

### `random.choice`
`random.choice(sequence)` picks one item from the sequence at random. Useful when you want to simulate unpredictable inputs.

### `input()`
`input("prompt")` pauses execution and waits for the user to type something and press Enter. In a Jupyter notebook, a text box appears below the cell. Type `low`, `medium`, or `high` and press Enter to continue.

### Why data engineers care
Input validation and comparison logic appear everywhere: checking that a user-supplied value is in the allowed set, that an agent's classification matches ground truth, or that a pipeline stage's output matches expected schema.

In [ ]:
import random

print("=== Charter Oak Mutual: Triage Faceoff ===")

levels = ["low", "medium", "high"]
rank = {"low": 1, "medium": 2, "high": 3}

actual = random.choice(levels)  # system assigns severity at random

# In Jupyter a text box appears below. Type low, medium, or high and press Enter.
your_call = input("Assess this claim: (low), (medium), or (high)? ").strip().lower()

if your_call not in rank:
    print("Not a valid severity. Choose low, medium, or high.")
elif your_call == actual:
    print(f"You assessed {your_call}. The system read {actual}.")
    print("Correct triage. Route it to the standard queue.")
elif rank[your_call] < rank[actual]:
    print(f"You assessed {your_call}. The system read {actual}.")
    print("Under-triaged. This claim is riskier than you flagged. Escalate it.")
else:
    print(f"You assessed {your_call}. The system read {actual}.")
    print("Over-triaged. You escalated more than needed. That ties up a senior adjuster.")

**Output explained**

Three outcomes are possible:
- **Correct** (`your_call == actual`): your assessment matched the system. Standard queue.
- **Under-triaged** (`rank[your_call] < rank[actual]`): you called it less severe than it is. The claim needs escalation you did not request.
- **Over-triaged** (`rank[your_call] > rank[actual]`): you called it more severe than it is. A senior adjuster's time was wasted on a routine claim.

The `rank` dictionary is a lookup table. Instead of writing nested `if` chains to compare severities, you translate each level to a number and use `<` / `>`. This pattern (map categorical values to ordinal numbers, then compare numerically) is extremely common in data pipelines.

---
## 04: Loops

### What it is
A `for` loop repeats a block of code for each item in a sequence. The sequence can be a string (iterates over characters), a list (iterates over elements), or a `range()` (iterates over integers).

**range(start, stop, step)**
- `range(5)` produces `0, 1, 2, 3, 4`
- `range(1, 6)` produces `1, 2, 3, 4, 5`
- `range(0, 10, 2)` produces `0, 2, 4, 6, 8`

### Why data engineers care
Loops are how you process every row in a file, every record in an API response, every item in a batch. In Weeks 3-5 you will see that SQL and PySpark abstract loops away, but the mental model is the same: apply an operation to each item.

In [ ]:
# Drill 04: Claim Pipeline Status Banner

stage = "INTAKE"

# Iterate over each character in the string
for letter in stage:
    print(f"Give me a {letter}!")
    print(f"{letter}!")

print()
print("What does that spell?!")
print(f"{stage}! Claims are moving through {stage}.")
print()

# range(): process a numbered batch of claims
print("Processing today's batch:")
for n in range(1, 6):
    print(f"  Claim {n} of 5 processed")

**Output explained**

The first loop iterates over the six characters in `"INTAKE"`. For each character it prints two lines, producing the cheer pattern.

The `range(1, 6)` loop generates the integers 1, 2, 3, 4, 5. Notice that `range` is exclusive on the right end: `range(1, 6)` does NOT include 6. This is consistent with Python's slice notation and list indexing.

**Pattern to remember**: `for i in range(1, n+1)` if you want 1 through n inclusive.

---
## 05: Lists

### What it is
A **list** is an ordered, mutable (changeable) collection. You can:
- **index** with `lst[i]` (zero-based; `-1` is the last element)
- **slice** with `lst[start:stop:step]`
- **append** with `lst.append(item)`
- **check length** with `len(lst)`

### Slicing quick reference
| Expression | Meaning |
|------------|---------|
| `lst[:2]` | first two elements |
| `lst[2:]` | everything after the first two |
| `lst[1::2]` | every other element starting at index 1 |

### Why data engineers care
Lists are the default container for a batch of records. You slice to take a sample, append to accumulate results, and iterate to process every record. DataFrames in pandas and PySpark are built on top of this concept.

In [ ]:
# Drill 05: Claim Queue Management

queue = ["CLM-1001", "CLM-1002", "CLM-1003", "CLM-1004",
         "CLM-1005", "CLM-1006", "CLM-1007"]

print("Today's claim queue:")
print(queue)
print()

print("First two claims:")
print(queue[:2])
print()

print("All claims except the first two:")
print(queue[2:])
print()

print("Every other claim, starting from the second:")
print(queue[1::2])
print()

# Append: a new claim arrives
queue.append("CLM-1008")
print("After appending CLM-1008:")
print(queue)
print()

# Update by index: a claim is reopened
queue[3] = "CLM-1004-REOPEN"
print("After marking CLM-1004 as reopened:")
print(queue)
print()

print(f"Total claims in queue: {len(queue)}")

**Output explained**

- `queue[:2]` returns `['CLM-1001', 'CLM-1002']`. The `:2` means "up to but not including index 2".
- `queue[2:]` returns everything from index 2 to the end.
- `queue[1::2]` starts at index 1 and steps by 2, picking every other element.
- `append` adds to the end; `len` counts all elements including the new one.
- Direct assignment `queue[3] = "..."` replaces the element at position 3 in place. The list grows or shrinks only with `append` / `remove`.

---
## 06: Dictionaries

### What it is
A **dictionary** maps unique **keys** to **values**. Any immutable type can be a key; strings are most common. Values can be anything.

```python
d = {"key1": value1, "key2": value2}
d["key1"]           # read
d["key1"] = new     # update existing
d["key_new"] = val  # add new key
del d["key1"]       # remove
```

### Why data engineers care
A dictionary is the shape of one row in a JSON API response. When you call an API and get back `{"claim_id": "CLM-1", "status": "open"}`, that is a Python dict. Knowing how to read, update, and delete keys is how you transform raw API data into clean records.

In [ ]:
# Drill 06: Claim Reserves Dictionary

reserves = {
    "CLM-2001": 32000, "CLM-2002": 28000, "CLM-2003": 17000,
    "CLM-2004": 12500, "CLM-2005": 8700,  "CLM-2006": 7200,
    "CLM-2007": 5300,  "CLM-2008": 4800,  "CLM-2009": 3100,
    "CLM-2010": 1200,  "CLM-2011": 950,   "CLM-2012": 400,
    "CLM-2013": 14500, "CLM-2014": 600,   "CLM-2015": 22000,
}

print(f"Entries before changes: {len(reserves)}")

# Re-estimate lowered the reserve on CLM-2003
reserves["CLM-2003"] = 16000
print(f"CLM-2003 reserve updated to ${reserves['CLM-2003']:,}")

# A new claim arrived
reserves["CLM-2016"] = 9000
print(f"CLM-2016 added with reserve ${reserves['CLM-2016']:,}")

# CLM-2013 was withdrawn
del reserves["CLM-2013"]
print("CLM-2013 removed (withdrawn)")

print(f"Entries after changes: {len(reserves)}")
print()

# Iterate over the dictionary
print("Current reserve book (sorted by claim id):")
for claim_id in sorted(reserves):
    print(f"  {claim_id}: ${reserves[claim_id]:>8,}")

**Output explained**

- Assigning to an existing key (`reserves["CLM-2003"] = 16000`) overwrites the value. No error; no warning. This is intentional.
- Assigning to a new key (`reserves["CLM-2016"] = 9000`) adds the key automatically.
- `del reserves["CLM-2013"]` removes the key-value pair entirely. The dictionary shrinks by 1.
- `sorted(reserves)` returns the keys in alphabetical order. Because claim IDs start with "CLM-" and are followed by a zero-padded number, alphabetical and numeric order agree here.
- `:>8,` right-aligns the number in an 8-character field with comma separation, making the amounts line up visually.

---
## 07: Iterating Lists and Accumulating Results

### What it is
You accumulate results by initializing a variable (counter, total, list) before the loop and updating it inside the loop. This pattern, called a **reduce** or **fold**, is the foundation for every aggregation you will do in SQL, pandas, and PySpark.

Common accumulators:
```python
total = 0
for x in items:
    total += x   # running sum
```

### `min` and `max` without built-ins
The drill below implements min and max from scratch to make the logic explicit. Later you would just use `min(items)` and `max(items)`.

In [ ]:
# Drill 07: Daily Claims Cash Flow

# 20 business days of net cash flow (recoveries minus payouts, in dollars)
daily_net = [-224,  352, 252, 354, -544,
             -650,   56, 123, -43,  254,
              325, -123,  47, 321,  123,
              133, -151, 613, 232, -311]

count = 0
total = 0
minimum = daily_net[0]
maximum = daily_net[0]
surplus_days = []
deficit_days = []

for day_net in daily_net:
    total += day_net
    count += 1

    # Track min / max manually to make the logic visible.
    # Two independent ifs, not if/elif: a single day can be a new
    # minimum or a new maximum, and both checks must run every time.
    if day_net < minimum:
        minimum = day_net
    if day_net > maximum:
        maximum = day_net

    if day_net > 0:
        surplus_days.append(day_net)
    else:
        deficit_days.append(day_net)

average = round(total / count, 2)

print("---------Summary Statistics----------")
print(f"Total days:       {count}")
print(f"Surplus days:     {len(surplus_days)} ({len(surplus_days)/count*100:.0f}%)")
print(f"Deficit days:     {len(deficit_days)} ({len(deficit_days)/count*100:.0f}%)")
print("-------------------------------------")
print(f"Surplus days:     {surplus_days}")
print(f"Deficit days:     {deficit_days}")
print("-------------------------------------")
print(f"Net cash flow:    ${total:,}")
print(f"Daily average:    ${average:,}")
print(f"Worst deficit:    ${minimum:,}")
print(f"Best surplus:     ${maximum:,}")

**Output explained**

The loop runs once per day. Each iteration:
1. Adds `day_net` to the running `total`.
2. Updates `minimum` and/or `maximum` if a new extreme is found.
3. Appends the value to `surplus_days` or `deficit_days`.

After the loop:
- `total / count` is the arithmetic mean.
- `len(surplus_days)` counts positive days without a separate counter.

**Why `minimum` and `maximum` start at `daily_net[0]`, not `0`**: a sentinel value of `0` only works if you also special-case the first iteration, and even then it breaks the moment a real value happens to equal the sentinel. Seeding both trackers with the first real data point removes the special case entirely; the first comparison in the loop is always a no-op compare against itself.

**Why two separate `if` statements, not `if / elif`**: with `elif`, only one branch can fire per iteration. If the first day in the list happens to be the largest value, an `elif`-chained version would spend that iteration only on the minimum check and silently skip recording the maximum, since the minimum branch already used up the only branch the elif chain allows. The bug would not raise an error: `maximum` would just end up wrong for that ordering. Two independent `if` statements let a single day update `minimum`, `maximum`, both, or neither, which is what the underlying logic actually requires.

**Result interpretation**: if total is positive, recoveries outpaced payouts over the month. The worst deficit day tells you the largest single-day shortfall, which drives reserve requirements.

---
## 08: Functions

### What it is
A **function** bundles a reusable piece of logic under a name. In Python:

```python
def function_name(param1, param2, param_with_default=value):
    result = ...
    return result
```

- `def` declares the function; the body is indented.
- Parameters inside the parentheses are the inputs.
- **Default parameters** make a parameter optional; callers can omit it.
- `return` sends a value back to the caller.

### When to use a function
Write a function when:
- You would copy-paste the same logic more than once.
- A block of code has a clear input-output contract.
- You want to give a piece of logic a meaningful name.

### Why data engineers care
Every transformation step in a pipeline is a function. Naming transformations makes pipelines readable, testable, and reusable.

In [ ]:
# Drill 08: Loss Ratio Calculator

def calculate_loss_ratio(incurred_losses, earned_premium, round_to=2):
    """Return the loss ratio as a percentage.

    Loss ratio = incurred losses / earned premium * 100
    Lower is better for the insurer.
    """
    loss_ratio = incurred_losses / earned_premium * 100
    return round(loss_ratio, round_to)


# Call the function for three consecutive years
year_2024 = calculate_loss_ratio(2900, 4500)
year_2025 = calculate_loss_ratio(3600, 4800)
year_2026 = calculate_loss_ratio(4200, 5000)

print(f"Loss Ratio 2024: {year_2024}%")
print(f"Loss Ratio 2025: {year_2025}%")
print(f"Loss Ratio 2026: {year_2026}%")

worst = max(year_2024, year_2025, year_2026)
print(f"Worst year loss ratio: {worst}%")

print()

# Challenge variant: accumulate results in a global list
loss_ratios = []

def calculate_loss_ratio_list(incurred_losses, earned_premium, round_to=2):
    """Append the loss ratio to the global list instead of returning it."""
    loss_ratio = incurred_losses / earned_premium * 100
    loss_ratios.append(round(loss_ratio, round_to))

calculate_loss_ratio_list(2900, 4500)
calculate_loss_ratio_list(3600, 4800)
calculate_loss_ratio_list(4200, 5000)

print("Loss ratios list:", loss_ratios)
print("Average loss ratio:", round(sum(loss_ratios) / len(loss_ratios), 2), "%")

**Output explained**

- `calculate_loss_ratio(2900, 4500)` computes `2900 / 4500 * 100 = 64.44%`.
- The `round_to=2` default means callers can write `calculate_loss_ratio(2900, 4500)` and get 2 decimal places automatically.
- 2026 has the highest loss ratio (84%), meaning the insurer paid out 84 cents for every dollar of premium earned. That is the worst year for profitability.

**Return vs. side-effect**: the first version returns a value (pure function, easy to test). The second version mutates a global list (side-effect). Both patterns exist in real code; returning a value is generally preferred because it is easier to test and reason about.

---
## 09: Nested Data Structures

### What it is
Real data is almost never flat. A REST API response looks like a dictionary that contains a list of dictionaries. This is **nested data**: containers inside containers.

Two common shapes:

**Dict of lists** (one claim, many payment events)
```python
{"CLM-3001": [["2026-06-01", "medical", 1200.0], ...]}
```

**Dict of dicts** (one mapping, many attributes per key)
```python
{"CLM-3001": {"date": "2026-06-22", "kind": "medical", "amount": 450.0}}
```

### Why data engineers care
JSON, the universal data exchange format, IS nested data. When you ingest an API response on Day 3, you will traverse exactly this structure. Drill 09 is the bridge between pure Python and API work.

In [ ]:
# Drill 09: Claim Payment Rollup

# Dict of list of records  (list = history of payments for one claim)
claim_payments = {
    "CLM-3001": [
        ["2026-06-01", "medical",  1200.00],
        ["2026-06-08", "rental",    300.00],
        ["2026-06-15", "medical",   800.00],
    ],
    "CLM-3002": [
        ["2026-06-02", "property", 2500.00],
        ["2026-06-12", "property", 1500.00],
    ],
    "CLM-3003": [
        ["2026-06-03", "medical",   600.00],
        ["2026-06-09", "rental",    400.00],
        ["2026-06-16", "medical",   250.00],
    ],
    "CLM-3004": [
        ["2026-06-05", "liability", 3000.00],
    ],
}

# Dict of dicts: one new payment per claim
new_payments = {
    "CLM-3001": {"date": "2026-06-22", "kind": "medical",   "amount": 450.00},
    "CLM-3002": {"date": "2026-06-22", "kind": "property",  "amount": 1000.00},
    "CLM-3003": {"date": "2026-06-22", "kind": "medical",   "amount": 150.00},
    "CLM-3004": {"date": "2026-06-22", "kind": "liability", "amount": 2000.00},
}

# Append each new payment to its claim's list
for claim_id, event in new_payments.items():
    record = [event["date"], event["kind"], event["amount"]]
    claim_payments[claim_id].append(record)

# Print a summary: total paid per claim
print("Payment summary after update:")
for claim_id, payments in claim_payments.items():
    total = sum(p[2] for p in payments)   # p[2] is the amount field
    print(f"  {claim_id}: {len(payments)} payments, total ${total:,.2f}")

**Output explained**

`new_payments.items()` gives pairs of `(claim_id, event_dict)`. Inside the loop:
1. We build a `[date, kind, amount]` list from the dict.
2. We append it to the existing list for that claim.

The summary loop then iterates over the updated `claim_payments`. The generator expression `sum(p[2] for p in payments)` is a compact way to add up the third element of every payment record without writing a separate loop.

**Bridge to Day 3**: when `requests.get(url).json()` returns a response, it is a Python dict or list with exactly this kind of nested structure. The skills you practiced here apply directly.

---
## 10: Classes

### What it is
A **class** is a blueprint for an object. An **object** bundles data (attributes) and the behavior that goes with that data (methods) into one unit.

```python
class MyClass:
    def __init__(self, arg1, arg2):   # called when you create an instance
        self.attr1 = arg1             # store data on self
        self.attr2 = arg2

    def my_method(self):              # a method: has self as first param
        return self.attr1 + self.attr2
```

### When to use a class vs. a function
| Use a class when... | Use a function when... |
|---------------------|----------------------|
| the same data is needed by multiple operations | you have one input, one output |
| you need to track state that changes over time | the operation is stateless |
| you are modeling a real-world entity | you are doing a one-off transformation |

### Why data engineers care
Classes model pipeline stages, API clients, and database connections. You will see classes extensively in SQLAlchemy, the Anthropic SDK, and AWS Boto3.

In [ ]:
# Drill 10: Claims as Classes

class Claim:
    def __init__(self, claim_id, policy_type, reserve, paid):
        self.claim_id = claim_id
        self.policy_type = policy_type
        self.reserve = reserve
        self.paid = paid

    def reserve_burn(self):
        """Percent of the reserve that has been paid out."""
        return round(self.paid / self.reserve * 100, 1)

    def is_over_reserve(self):
        """True once payments exceed the reserve."""
        return self.paid > self.reserve

    def add_payment(self, amount):
        """Post a new payment. The object's state changes."""
        self.paid += amount

    def summary(self):
        flag = " OVER RESERVE" if self.is_over_reserve() else ""
        return f"{self.claim_id} ({self.policy_type}): burn {self.reserve_burn()}%{flag}"


# Create instances
claims = [
    Claim("CLM-6001", "auto",      5000.0,  1200.0),
    Claim("CLM-6002", "property", 12000.0, 13500.0),
    Claim("CLM-6003", "liability", 20000.0,  4000.0),
]

print("Initial state:")
for c in claims:
    print(" ", c.summary())

# A payment posts to the first claim
claims[0].add_payment(4200.0)
print(f"\nAfter a $4,200 payment on CLM-6001:")
print(" ", claims[0].summary())

over = sum(1 for c in claims if c.is_over_reserve())
print(f"\nClaims over reserve: {over} of {len(claims)}")

**Output explained**

- `CLM-6001` starts at 24% burn (1200 / 5000). After the $4,200 payment, total paid is $5,400, which exceeds the $5,000 reserve. It is now marked `OVER RESERVE`.
- `CLM-6002` was already over reserve before any changes (13500 > 12000).
- `CLM-6003` is only at 20% burn and well within reserve.

**Key OOP insight**: `add_payment` changes `self.paid` on the specific claim object. Each `Claim` instance carries its own independent state. When you call `claims[0].add_payment(...)`, only `claims[0]` changes. `claims[1]` and `claims[2]` are unaffected.

The generator `sum(1 for c in claims if c.is_over_reserve())` counts True results from calling the method on every claim. This is the "count how many records pass a condition" pattern you will use constantly in pandas.

---
## 11: File I/O

### What it is
File I/O (input/output) is how Python reads from and writes to files on disk. The canonical pattern uses a **context manager** (`with` statement) to open a file and guarantee it is closed when you are done, even if an error occurs.

```python
with open("path/to/file.txt") as f:   # open for reading (default)
    for line in f:
        ...

with open("output.txt", "w") as f:    # open for writing
    f.write("some text\n")
```

### `pathlib.Path`
`Path("relative/path")` builds a platform-safe file path. Use it instead of string concatenation for file paths.

### Why data engineers care
Before pandas, you read raw files with plain Python. File I/O is also how you write log files, audit trails, and flat-file outputs from pipelines.

In [ ]:
import io
from pathlib import Path

# In a notebook we create the data inline using io.StringIO
# In a real project this would be: open(Path("Resources/daily_claims.txt")) as f
raw_data = """42
55
38
61
47
50
44
58
39
52
49
63
41
57
45
53
48
60
43
55"""

# --- Reading ---
total = 0
day_count = 0

# io.StringIO wraps a string so it behaves like a file object
with io.StringIO(raw_data) as f:
    for line in f:
        line = line.strip()
        if line == "":
            continue
        total += int(line)
        day_count += 1

daily_average = round(total / day_count, 2)

print(f"Total claims processed: {total}")
print(f"Days in period:         {day_count}")
print(f"Daily average:          {daily_average}")

# --- Writing ---
# Write a summary file to disk (will appear in the same folder as this notebook)
OUT = Path("claims_summary.txt")
with open(OUT, "w") as f:
    f.write(f"Total claims: {total}\n")
    f.write(f"Days: {day_count}\n")
    f.write(f"Daily average: {daily_average}\n")

print(f"\nSummary written to {OUT}")

# Verify by reading it back
with open(OUT) as f:
    print("\nFile contents:")
    print(f.read())

**Output explained**

- `io.StringIO(raw_data)` wraps the multi-line string so we can iterate over it with the same `for line in f` pattern we would use on a real file. The behavior is identical.
- `.strip()` removes the trailing `\n` from each line before we call `int()`.
- The `if line == ""` guard skips blank lines (common in real data files).
- The second block writes three lines to disk with explicit `\n` newline characters. On the next open, `f.read()` returns the entire file as one string.

**Mode reminder**: `open(path)` defaults to `"r"` (read). Use `"w"` to write (overwrites), `"a"` to append (adds to the end).

---
## 12: CSV I/O

### What it is
CSV (comma-separated values) is the simplest tabular format. Python's `csv` module handles the edge cases (quoted fields, embedded commas) that plain `split(",")` misses.

Key classes:
- `csv.DictReader(f)` turns each row into a `dict` keyed by the header. This is usually what you want.
- `csv.DictWriter(f, fieldnames=...)` writes rows from dicts with a header.

### Why data engineers care
CSV is the most common flat-file format in insurance, finance, and government data. On Day 4 you will use pandas `read_csv()`, which is faster and more powerful, but the `csv` module makes the mechanics explicit.

In [ ]:
import csv, io

# Inline CSV data (same as Resources/claims.csv in the drill folder)
csv_data = """claim_id,policy_type,reserve,paid
CLM-7001,auto,5000,1200
CLM-7002,property,12000,11800
CLM-7003,auto,3000,500
CLM-7004,liability,20000,4000
CLM-7005,property,15000,16200
CLM-7006,auto,4500,4500
CLM-7007,liability,11000,9800
CLM-7008,property,9000,2200
CLM-7009,auto,6200,3100
CLM-7010,liability,18000,2500"""

# --- Read and aggregate ---
summary = {}

with io.StringIO(csv_data) as f:
    reader = csv.DictReader(f)
    for row in reader:
        ptype = row["policy_type"]
        reserve = float(row["reserve"])
        paid = float(row["paid"])
        bucket = summary.setdefault(ptype, {"count": 0, "reserve": 0.0, "paid": 0.0})
        bucket["count"] += 1
        bucket["reserve"] += reserve
        bucket["paid"] += paid

print("Policy type summary:")
for ptype in sorted(summary):
    b = summary[ptype]
    loss_ratio = round(b["paid"] / b["reserve"] * 100, 1)
    print(f"  {ptype:10s}: {b['count']} claims | "
          f"reserve ${b['reserve']:>10,.2f} | "
          f"paid ${b['paid']:>10,.2f} | "
          f"loss ratio {loss_ratio}%")

# --- Write to a CSV file ---
from pathlib import Path
OUT = Path("policy_summary.csv")

with open(OUT, "w", newline="") as f:
    writer = csv.DictWriter(
        f, fieldnames=["policy_type", "count", "total_reserve", "total_paid"]
    )
    writer.writeheader()
    for ptype in sorted(summary):
        b = summary[ptype]
        writer.writerow({
            "policy_type": ptype,
            "count": b["count"],
            "total_reserve": round(b["reserve"], 2),
            "total_paid": round(b["paid"], 2),
        })

print(f"\nWrote {OUT}")

**Output explained**

`csv.DictReader` gives each row as a dict: `{"claim_id": "CLM-7001", "policy_type": "auto", "reserve": "5000", "paid": "1200"}`. All values are strings. You must call `float()` or `int()` to convert numeric fields.

`setdefault(ptype, {...})` is a dict trick: if `ptype` is not yet a key, insert the default dict and return it. If it already exists, return the existing value. This avoids the `if ptype not in summary: summary[ptype] = {...}` boilerplate.

The loss ratio column shows which policy types are most costly:
- `property` and `liability` with loss ratios near or above 80% are flagging potential underwriting issues.
- `auto` with a mix of ratios is more balanced.

`newline=""` in `open(..., "w", newline="")` is required for `csv.DictWriter` on Windows to avoid double line endings. It is harmless on Linux.

---
## 13: JSON I/O

### What it is
JSON (JavaScript Object Notation) is the universal format for API responses and web data. Python's `json` module converts between JSON text and Python objects:

| JSON type | Python type |
|-----------|-------------|
| `object {}` | `dict` |
| `array []` | `list` |
| `string ""` | `str` |
| `number` | `int` or `float` |
| `true / false` | `True / False` |
| `null` | `None` |

```python
json.loads(string)    # parse a JSON string into Python
json.load(file)       # parse a JSON file into Python
json.dumps(obj)       # serialize Python to a JSON string
json.dump(obj, file)  # serialize Python to a JSON file
```

### Why data engineers care
Every REST API returns JSON. When you call an API with `requests.get(url).json()` on Day 3, you will get back a Python dict/list and traverse it exactly as you do here.

In [ ]:
import json
from pathlib import Path

# Inline JSON (same as Resources/claims.json in the drill folder)
raw_json = """[
  {"claim_id": "CLM-8001", "policy_type": "auto",
   "payments": [{"kind": "medical", "amount": 800},
                {"kind": "rental",  "amount": 400}]},
  {"claim_id": "CLM-8002", "policy_type": "property",
   "payments": [{"kind": "property", "amount": 5000},
                {"kind": "property", "amount": 2500}]},
  {"claim_id": "CLM-8003", "policy_type": "liability",
   "payments": []},
  {"claim_id": "CLM-8004", "policy_type": "auto",
   "payments": [{"kind": "medical", "amount": 1500}]},
  {"claim_id": "CLM-8005", "policy_type": "property",
   "payments": [{"kind": "property", "amount": 3200},
                {"kind": "medical",  "amount": 800}]}
]"""

# Parse: JSON string -> Python list of dicts
claims = json.loads(raw_json)

print("Claim payment totals:")
totals = []
for claim in claims:
    total = sum(p["amount"] for p in claim["payments"])
    totals.append({"claim_id": claim["claim_id"], "total_paid": total})
    print(f"  {claim['claim_id']}: ${total:,}")

# Serialize: Python object -> JSON string (pretty-printed with indent)
OUT = Path("claims_totals.json")
with open(OUT, "w") as f:
    json.dump(totals, f, indent=2)

print(f"\nWrote {OUT}")
print("\nJSON output preview:")
print(json.dumps(totals, indent=2))

**Output explained**

- `json.loads(raw_json)` parses the JSON string into a Python list of dicts. After this line, `claims` is identical to what `requests.get(url).json()` returns on Day 3.
- `sum(p["amount"] for p in claim["payments"])` iterates over the nested payments list and sums the "amount" field. `CLM-8003` has an empty payments list, so its total is 0.
- `json.dump(totals, f, indent=2)` writes pretty-printed JSON: each key-value pair on its own line, indented by 2 spaces. This format is human-readable and commonly used for configuration files and audit logs.

**Important**: `json.dump` vs `json.dumps`: the one with the `s` works with strings, the one without works with files.

---
## 14: Lambdas, map(), and filter()

### What it is
A **lambda** is a small anonymous function defined in one expression.

```python
lambda x: x * 2        # a function that doubles x
lambda c: c["paid"]    # extracts the "paid" field from a dict
```

**`map(fn, iterable)`** applies `fn` to every item and returns an iterator. Wrap in `list()` to materialize the results.

**`filter(fn, iterable)`** keeps only the items where `fn(item)` is `True`.

### Why data engineers care
In Week 5 (PySpark), `rdd.map(lambda row: ...)` and `rdd.filter(lambda row: ...)` are exactly how you transform and filter distributed datasets. Learning the pattern here with plain Python makes the PySpark API feel familiar.

In [ ]:
# Drill 14: Lambdas, map, and filter

claims = [
    {"claim_id": "CLM-9001", "policy_type": "auto",      "reserve": 5000,  "paid": 1200, "status": "open"},
    {"claim_id": "CLM-9002", "policy_type": "property",  "reserve": 12000, "paid": 11800,"status": "closed"},
    {"claim_id": "CLM-9003", "policy_type": "liability", "reserve": 20000, "paid": 4000, "status": "open"},
    {"claim_id": "CLM-9004", "policy_type": "auto",      "reserve": 3000,  "paid": 500,  "status": "open"},
    {"claim_id": "CLM-9005", "policy_type": "property",  "reserve": 15000, "paid": 16200,"status": "open"},
]

# sort key: sort by reserve descending
by_reserve = sorted(claims, key=lambda c: c["reserve"], reverse=True)
print("Sorted by reserve (desc):", [c["claim_id"] for c in by_reserve])

# map: project each claim to just its id
ids = list(map(lambda c: c["claim_id"], claims))
print("All ids:", ids)

# map with transform: bump every reserve by 10%
bumped = list(map(lambda c: round(c["reserve"] * 1.10, 2), claims))
print("Reserves +10%:", bumped)

# filter: keep only open claims
open_claims = list(filter(lambda c: c["status"] == "open", claims))
print("Open claim ids:", [c["claim_id"] for c in open_claims])

# filter + map: ids of claims where paid > reserve (reserve breach)
breaches = list(map(
    lambda c: c["claim_id"],
    filter(lambda c: c["paid"] > c["reserve"], claims)
))
print("Reserve breaches:", breaches)

**Output explained**

- **sorted with key**: the lambda extracts the `reserve` field to use as the sort key. `reverse=True` gives descending order. `sorted()` returns a new list; the original is unchanged.
- **map ids**: projects each dict down to just its `claim_id`. The result is a flat list of strings, not a list of dicts.
- **map +10%**: applies an arithmetic transform to each reserve value.
- **filter open**: keeps only dicts where `status == "open"`. `CLM-9002` (closed) is excluded.
- **filter + map combined**: first filter keeps breaches (`paid > reserve`), then map extracts their ids. Only `CLM-9005` has `paid = 16200 > reserve = 15000`.

**Modern Python note**: list comprehensions often replace `map` + `filter` for readability. Section A1 (Comprehensions) shows the equivalent comprehension syntax.

---
## 15: SQLite

### What it is
SQLite is a full SQL database engine built into Python's standard library. No server, no install. The entire database is a single file (or `:memory:` for an in-memory database that vanishes when the connection closes).

```python
import sqlite3
conn = sqlite3.connect(":memory:")   # or "myfile.db" for a file
cur = conn.cursor()
cur.execute("SELECT ...")
conn.commit()
conn.close()
```

**Parameterized queries** (`?` placeholders) prevent SQL injection:
```python
cur.execute("SELECT * FROM t WHERE id = ?", ("CLM-1",))
```

### Why data engineers care
You will use SQL every day in BigQuery, Athena, and Snowflake. SQLite lets you practice SQL without a cloud account. The `SELECT`, `GROUP BY`, and `ORDER BY` syntax is identical.

In [ ]:
import sqlite3

# Use an in-memory database (no file needed, no cleanup required)
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# Create the table
cur.execute("""
    CREATE TABLE claims (
        claim_id    TEXT PRIMARY KEY,
        policy_type TEXT,
        reserve     REAL,
        paid        REAL
    )
""")

# Insert rows with parameterized queries (safe, never use string formatting for SQL)
rows = [
    ("CLM-1001", "auto",      5000.0,  1200.0),
    ("CLM-1002", "property", 12000.0, 11800.0),
    ("CLM-1003", "liability",20000.0,  4000.0),
    ("CLM-1004", "auto",      3000.0,   500.0),
    ("CLM-1005", "property", 15000.0, 16200.0),
]
cur.executemany("INSERT INTO claims VALUES (?, ?, ?, ?)", rows)
conn.commit()

# Filtered SELECT
print("Auto claims:")
for row in cur.execute(
    "SELECT claim_id, reserve FROM claims WHERE policy_type = ?", ("auto",)
):
    print(f"  {row[0]}: ${row[1]:,.2f}")

# Aggregate with GROUP BY
print("\nTotal reserve by policy type:")
for row in cur.execute(
    "SELECT policy_type, SUM(reserve), SUM(paid), "
    "ROUND(SUM(paid) / SUM(reserve) * 100, 1) AS loss_ratio "
    "FROM claims GROUP BY policy_type ORDER BY policy_type"
):
    print(f"  {row[0]:10s}: reserve ${row[1]:>10,.2f} | paid ${row[2]:>10,.2f} | loss ratio {row[3]}%")

conn.close()

**Output explained**

- The `?` placeholders in `INSERT INTO claims VALUES (?, ?, ?, ?)` are filled by the tuple values in order. This is the safe, parameterized way. Never use f-strings or string concatenation to build SQL, as that opens the door to SQL injection.
- `executemany` runs the same INSERT statement once per row in the list, which is more efficient than calling `execute` in a loop.
- The `GROUP BY policy_type` query produces one row per type and aggregates all claims in that group. `ROUND(SUM(paid) / SUM(reserve) * 100, 1)` is an inline loss ratio calculation done entirely in SQL.

**Bridge to Week 3**: in BigQuery you will write the same `SELECT ... GROUP BY ... ORDER BY` pattern against millions of rows. The SQL syntax is identical; only the engine changes.

---
## 16: Error Handling

### What it is
Production data is messy. Files have missing fields. APIs return garbage. Users type invalid input. **Exception handling** lets your code recover gracefully instead of crashing.

```python
try:
    result = risky_operation()
except SomeError as exc:
    # runs only if SomeError was raised
    handle(exc)
else:
    # runs only if NO exception was raised
    use(result)
finally:
    # ALWAYS runs, with or without an exception
    cleanup()
```

**Custom exceptions** (`class MyError(ValueError): pass`) let you distinguish your own errors from library errors and give callers a specific type to catch.

### Why data engineers care
API ingestion, file reading, and database writes all fail in the real world. A pipeline that crashes on the first bad record is useless. Robust pipelines catch errors, log them, and keep processing.

In [ ]:
import random

# ---- Custom exception ----
class InvalidAmountError(ValueError):
    """Raised when an amount field is missing or not a valid non-negative number."""


def parse_amount(raw):
    """Convert raw string to a non-negative float, or raise InvalidAmountError."""
    try:
        value = float(raw)
    except (TypeError, ValueError):
        raise InvalidAmountError(f"cannot parse amount: {raw!r}")
    if value < 0:
        raise InvalidAmountError(f"negative amount: {value}")
    return round(value, 2)


# ---- Core: process a batch with mixed data quality ----
raw_claims = [
    {"claim_id": "CLM-1", "amount": "1200.50"},
    {"claim_id": "CLM-2", "amount": "N/A"},    # non-numeric
    {"claim_id": "CLM-3"},                      # missing 'amount' key
    {"claim_id": "CLM-4", "amount": "-50"},     # negative
]

clean = []
errors = 0

for claim in raw_claims:
    cid = claim["claim_id"]
    try:
        amount = parse_amount(claim.get("amount"))  # .get() returns None if key missing
    except InvalidAmountError as exc:
        print(f"  REJECTED {cid}: {exc}")
        errors += 1
        continue
    else:
        clean.append({"claim_id": cid, "amount": amount})

print(f"\nClean records: {clean}")
print(f"Rejected records: {errors}")

# ---- Challenge: retry logic (foreshadows Day 3 API retries) ----
def flaky_fetch(claim_id):
    if random.random() < 0.5:
        raise ConnectionError("temporary network error")
    return {"claim_id": claim_id, "status": "ok"}

def fetch_with_retry(claim_id, max_tries=3):
    for attempt in range(1, max_tries + 1):
        try:
            return flaky_fetch(claim_id)
        except ConnectionError as exc:
            print(f"  attempt {attempt} failed: {exc}")
    raise RuntimeError(f"gave up on {claim_id} after {max_tries} tries")

print("\nRetry demo:")
try:
    result = fetch_with_retry("CLM-5")
    print(f"  Success: {result}")
except RuntimeError as exc:
    print(f"  Exhausted retries: {exc}")

**Output explained**

- `CLM-1` succeeds: `"1200.50"` converts cleanly to `1200.5`.
- `CLM-2` fails: `float("N/A")` raises `ValueError`, caught as `InvalidAmountError`.
- `CLM-3` fails: `.get("amount")` returns `None` (key is missing). `float(None)` raises `TypeError`, also caught as `InvalidAmountError`.
- `CLM-4` fails: `-50` is a valid float but negative. The explicit check raises the error.

**Pattern**: `try` / `except SpecificError` / `continue` is the standard way to skip a bad record and keep processing. Broad `except Exception` is a code smell; always catch the most specific exception you can.

**Retry demo**: `flaky_fetch` fails 50% of the time (simulates a real API under load). `fetch_with_retry` attempts up to 3 times before giving up. You will build this pattern properly on Day 3 with the `httpx` and `tenacity` libraries.

---
## 17: The collections Module

### What it is
The `collections` standard library module provides specialized containers that solve common aggregation patterns cleanly.

| Container | What it does |
|-----------|-------------|
| `Counter(iterable)` | counts occurrences of each unique value |
| `defaultdict(default_factory)` | a dict where missing keys get a default value automatically |

### `Counter`
```python
from collections import Counter
Counter(["a", "b", "a", "c", "b", "a"])
# -> Counter({'a': 3, 'b': 2, 'c': 1})
```

### `defaultdict(float)` vs. `dict.setdefault`
```python
from collections import defaultdict
d = defaultdict(float)
d["key"] += 1.5    # works even though "key" was never inserted
```

### Why data engineers care
`Counter` and `defaultdict` replace the verbose "check if key exists, create if not, then update" pattern that beginners write. On Day 4, `pandas.groupby` does the same thing at scale.

In [ ]:
from collections import defaultdict, Counter

claims = [
    {"claim_id": "CLM-1", "policy_type": "auto",      "paid": 1200.0},
    {"claim_id": "CLM-2", "policy_type": "property",  "paid": 5000.0},
    {"claim_id": "CLM-3", "policy_type": "auto",      "paid":  800.0},
    {"claim_id": "CLM-4", "policy_type": "liability", "paid": 4000.0},
    {"claim_id": "CLM-5", "policy_type": "property",  "paid": 2200.0},
    {"claim_id": "CLM-6", "policy_type": "auto",      "paid": 3100.0},
]

# Counter: tally claim counts by policy type in one line
counts = Counter(c["policy_type"] for c in claims)
print("Claim counts:", dict(counts))
print("Most common:", counts.most_common(1))

# defaultdict(float): sum paid by policy type without setdefault
paid_by_type = defaultdict(float)
for c in claims:
    paid_by_type[c["policy_type"]] += c["paid"]
print("\nTotal paid by type:", dict(paid_by_type))

# defaultdict(list): group claim ids by policy type
ids_by_type = defaultdict(list)
for c in claims:
    ids_by_type[c["policy_type"]].append(c["claim_id"])
print("Claim ids by type:", dict(ids_by_type))

# Average paid per type (combine Counter and defaultdict)
print("\nAverage paid by type:")
for ptype in sorted(paid_by_type):
    avg = round(paid_by_type[ptype] / counts[ptype], 2)
    print(f"  {ptype:10s}: ${avg:,.2f}")

**Output explained**

- `Counter(c["policy_type"] for c in claims)` iterates the generator and tallies each value. Result: `{"auto": 3, "property": 2, "liability": 1}`.
- `counts.most_common(1)` returns a list of the top N (count, value) pairs: `[("auto", 3)]`.
- `defaultdict(float)` starts every new key at `0.0` automatically, so `paid_by_type["auto"] += 1200.0` works on the very first access without `KeyError`.
- Combining `paid_by_type[ptype]` (sum) and `counts[ptype]` (count) gives the average.

**Bridge to Day 4**: `pandas.groupby("policy_type")["paid"].mean()` does this in one line on a DataFrame. Understanding the manual version helps you debug the pandas version.

---
## 18: datetime

### What it is
Time is everywhere in data engineering: claim opened/closed dates, taxi timestamps, API event times. Python's `datetime` module handles parsing, formatting, and arithmetic.

Key classes:
- `datetime` (from `datetime`): a point in time (date + time)
- `timedelta`: a duration (difference between two datetimes)

```python
from datetime import datetime, timedelta

# Parse a string into a datetime
dt = datetime.strptime("2026-05-02", "%Y-%m-%d")

# Format a datetime into a string
s = dt.strftime("%m/%d/%Y")

# Date arithmetic
due = dt + timedelta(days=30)
duration = closed - opened   # timedelta; use .days for the integer
```

### Why data engineers care
Every timestamp in a pipeline must be parsed, validated, and sometimes shifted (for time zones or business-day calculations). On Day 4, pandas will do most of this with `pd.to_datetime()`, but the underlying concepts are identical.

In [ ]:
from datetime import datetime, timedelta

claims = [
    {"claim_id": "CLM-1", "opened": "2026-05-02", "closed": "2026-05-20"},
    {"claim_id": "CLM-2", "opened": "2026-05-10", "closed": "2026-06-01"},
    {"claim_id": "CLM-3", "opened": "2026-05-15", "closed": "2026-05-18"},
]

# Parse a string into a datetime
opened = datetime.strptime(claims[0]["opened"], "%Y-%m-%d")
print("Parsed:", opened)
print("Year:", opened.year, "| Weekday:", opened.strftime("%A"))

# Format a datetime back to a string
print("Formatted:", opened.strftime("%m/%d/%Y"))

# timedelta: days each claim was open
print("\nDays open per claim:")
for c in claims:
    o  = datetime.strptime(c["opened"], "%Y-%m-%d")
    cl = datetime.strptime(c["closed"], "%Y-%m-%d")
    print(f"  {c['claim_id']}: {(cl - o).days} days")

# Sort claims by opened date
by_date = sorted(claims, key=lambda c: datetime.strptime(c["opened"], "%Y-%m-%d"))
print("\nBy opened date:", [c["claim_id"] for c in by_date])

# Date arithmetic: follow-up due 30 days after opening
due = opened + timedelta(days=30)
print("Follow-up due:", due.strftime("%Y-%m-%d"))

# Challenge: average days open and longest-running claim
durations = {
    c["claim_id"]: (datetime.strptime(c["closed"], "%Y-%m-%d")
                    - datetime.strptime(c["opened"], "%Y-%m-%d")).days
    for c in claims
}
avg = round(sum(durations.values()) / len(durations), 1)
longest = max(durations, key=durations.get)
print(f"\nAverage days open: {avg}")
print(f"Longest-running claim: {longest} ({durations[longest]} days)")

**Output explained**

- `strptime("2026-05-02", "%Y-%m-%d")` parses the ISO 8601 date string. The format codes: `%Y` = 4-digit year, `%m` = 2-digit month, `%d` = 2-digit day.
- Subtracting two `datetime` objects gives a `timedelta`. The `.days` attribute extracts the integer number of days.
- `sorted(..., key=lambda c: datetime.strptime(...))` sorts by parsed date, not alphabetically by string. Alphabetical and date order agree for ISO 8601 format, but using `strptime` in the key is the safe, explicit approach.
- The challenge dict comprehension builds `{claim_id: days_open}` for all three claims in one expression, then `max(..., key=durations.get)` returns the key with the highest value.

---
## Advanced Track: A1 - Comprehensions

### What it is
A **comprehension** is a compact expression that builds a list, dict, or set by applying an expression to each item in an iterable, with an optional filter.

| Type | Syntax |
|------|--------|
| list | `[expr for item in iterable if condition]` |
| dict | `{key_expr: val_expr for item in iterable if condition}` |
| set | `{expr for item in iterable if condition}` |

Comprehensions replace many simple `for` loops with one readable line. They are idiomatic Python and you will see them everywhere in data engineering code.

### Why the filter clause matters
`[c["claim_id"] for c in claims if c["status"] == "open"]` is equivalent to:
```python
result = []
for c in claims:
    if c["status"] == "open":
        result.append(c["claim_id"])
```

### In PySpark
PySpark's `DataFrame.select()` and `DataFrame.filter()` are the distributed equivalents of list and set comprehensions.

In [ ]:
# Advanced A1: Comprehensions

claims = [
    {"claim_id": "CLM-4001", "policy_type": "auto",      "status": "open",   "state": "CT", "reserve": 5000.0,  "paid": 1200.0},
    {"claim_id": "CLM-4002", "policy_type": "property",  "status": "closed", "state": "MA", "reserve": 12000.0, "paid": 11800.0},
    {"claim_id": "CLM-4003", "policy_type": "auto",      "status": "denied", "state": "CT", "reserve": 8000.0,  "paid": 0.0},
    {"claim_id": "CLM-4004", "policy_type": "liability", "status": "open",   "state": "RI", "reserve": 28000.0, "paid": 4000.0},
    {"claim_id": "CLM-4005", "policy_type": "property",  "status": "open",   "state": "CT", "reserve": 15000.0, "paid": 16200.0},
    {"claim_id": "CLM-4006", "policy_type": "auto",      "status": "open",   "state": "MA", "reserve": 3000.0,  "paid": 500.0},
]

# 1. List comprehension: every claim id
all_ids = [c["claim_id"] for c in claims]
print("All ids:", all_ids)

# 2. Filtered list: ids of open claims only
open_ids = [c["claim_id"] for c in claims if c["status"] == "open"]
print("Open ids:", open_ids)

# 3. Dict comprehension: claim_id -> reserve lookup table
reserve_by_id = {c["claim_id"]: c["reserve"] for c in claims}
print("Reserve by id:", reserve_by_id)

# 4. Set comprehension: distinct policy types
policy_types = {c["policy_type"] for c in claims}
print("Unique policy types:", sorted(policy_types))

# 5. Dict comprehension with transform and filter: burn % for claims with payments
burn_by_id = {
    c["claim_id"]: round(c["paid"] / c["reserve"] * 100, 1)
    for c in claims
    if c["paid"] > 0
}
print("Burn % (claims with payments):", burn_by_id)

# Challenge A: nested comprehension - flatten all payment amounts
payments = {
    "CLM-4001": [["2026-06-01", "medical", 800.0], ["2026-06-08", "rental", 400.0]],
    "CLM-4004": [["2026-06-05", "liability", 4000.0]],
    "CLM-4005": [["2026-06-02", "property", 16200.0]],
}
all_amounts = [rec[2] for events in payments.values() for rec in events]
print("All payment amounts:", all_amounts)

# Challenge B: conditional expression inside comprehension
labels = {
    c["claim_id"]: ("over" if c["paid"] > c["reserve"] else "within")
    for c in claims
}
print("Reserve status:", labels)

**Output explained**

- The list comprehension in step 2 applies the filter `if c["status"] == "open"` inline. Claims with `status = "closed"` or `"denied"` are excluded.
- The dict comprehension in step 3 creates a lookup table. You can now find any reserve in O(1) time: `reserve_by_id["CLM-4001"]` -> `5000.0`.
- The set comprehension in step 4 deduplicates automatically. Sets have no order, so `sorted()` is needed for stable output.
- The burn % comprehension skips `CLM-4003` (paid = 0.0) with the `if c["paid"] > 0` filter.

**Nested comprehension** in Challenge A: `for events in payments.values()` iterates over each claim's payment list, and the inner `for rec in events` iterates each individual payment. `rec[2]` picks the amount field. The result is a single flat list of all amounts.

**Ternary (inline conditional)** in Challenge B: `"over" if ... else "within"` is Python's one-liner equivalent of `if/else`. Use it inside comprehensions when the condition is simple.

---
## Advanced Track: A2 - Structural Pattern Matching

### What it is
`match / case` (Python 3.10+) reads the **structure** of a value, not just a single equality check. It can match dictionary keys, list shapes, tuple patterns, and value guards.

```python
match some_dict:
    case {"status": "denied"}:
        ...
    case {"policy_type": "auto", "reserve": reserve} if reserve > 25000:
        ...   # guard: extra condition after the pattern
    case _:
        ...   # wildcard: matches anything not caught above
```

### Why data engineers care
API responses and config files are nested dicts. `match / case` is the cleanest way to route a record based on its shape without deeply nested `if / elif` chains.

In [ ]:
# Advanced A2: Structural Pattern Matching (Python 3.10+)

claims = [
    {"claim_id": "CLM-4001", "policy_type": "auto",      "status": "open",   "reserve": 5000.0},
    {"claim_id": "CLM-4002", "policy_type": "property",  "status": "closed", "reserve": 12000.0},
    {"claim_id": "CLM-4003", "policy_type": "auto",      "status": "denied", "reserve": 8000.0},
    {"claim_id": "CLM-4004", "policy_type": "liability", "status": "open",   "reserve": 28000.0},
    {"claim_id": "CLM-4005", "policy_type": "property",  "status": "open",   "reserve": 15000.0},
    {"claim_id": "CLM-4006", "policy_type": "auto",      "status": "open",   "reserve": 3000.0},
    {"claim_id": "CLM-4007", "policy_type": "auto",      "status": "open",   "reserve": 30000.0},
]


def route_claim(claim):
    match claim:
        case {"status": "denied"}:
            return "closed"
        case {"policy_type": "auto", "reserve": reserve} if reserve > 25000:
            return "SIU"
        case {"status": "open", "reserve": reserve} if reserve >= 20000:
            return "senior adjuster"
        case {"status": "open"}:
            return "standard queue"
        case _:
            return "manual review"


print("Claim routing:")
for c in claims:
    print(f"  {c['claim_id']}: {route_claim(c)}")


# Destructuring a nested list record
def describe_payment(record):
    match record:
        case [date, kind, amount]:
            return f"{date}: {kind} payment of ${amount:,.2f}"
        case _:
            return "unknown record format"


print("\nPayment records:")
records = [
    ["2026-06-01", "medical",  800.0],
    ["2026-06-05", "liability", 4000.0],
    ["bad record"],
]
for r in records:
    print(f"  {describe_payment(r)}")


# Tuple patterns with OR and wildcard
def priority(status, policy_type):
    match (status, policy_type):
        case ("denied", _):
            return "none"
        case ("open", "liability"):
            return "high"
        case ("open", "auto" | "property"):
            return "normal"
        case _:
            return "low"


print("\nPriority routing:")
for c in claims:
    print(f"  {c['claim_id']}: {priority(c['status'], c['policy_type'])}")

**Output explained**

Cases are evaluated top to bottom; the first match wins.
- `CLM-4003` matches `{"status": "denied"}` immediately: routed to "closed".
- `CLM-4007` matches the second case: policy_type is "auto" AND reserve is 30000 > 25000. Routed to SIU.
- `CLM-4004` (liability, reserve 28000) does NOT match the second case (policy_type is not "auto"), but DOES match the third case (status is "open" and reserve >= 20000): senior adjuster.

**Guard clause** (`if reserve > 25000`): a pattern can include a boolean guard after `if`. The pattern must match AND the guard must be True for the case to fire.

**Destructuring** in `describe_payment`: `case [date, kind, amount]` matches any list of exactly three elements and binds the elements to those variable names. `["bad record"]` is a list of one element, so it falls through to `case _`.

**`|` (OR) in tuple pattern**: `"auto" | "property"` matches either value. This is specific to `match / case` syntax, not the bitwise OR operator.

---
## Advanced Track: A3 - Classes and Dataclasses

### What it is
A **dataclass** (Python 3.7+) is a decorator that auto-generates `__init__`, `__repr__`, and `__eq__` from typed field declarations. It eliminates the boilerplate you saw in Drill 10 while adding type hints.

```python
from dataclasses import dataclass

@dataclass
class Claim:
    claim_id: str
    reserve: float
    paid: float = 0.0   # default value
```

This is equivalent to writing `__init__`, `__repr__`, and `__eq__` by hand.

### Type hints
Type hints (`name: str`, `amount: float`) document what type a variable is expected to hold. Python does NOT enforce them at runtime, but tools like `mypy` and IDEs use them for static checking and autocompletion.

In [ ]:
from dataclasses import dataclass


# ---- Part 1: Plain class (you write all boilerplate) ----
class ClaimPlain:
    def __init__(self, claim_id, policy_type, reserve, paid):
        self.claim_id = claim_id
        self.policy_type = policy_type
        self.reserve = reserve
        self.paid = paid

    def reserve_burn(self):
        return round(self.paid / self.reserve * 100, 1)

    def __repr__(self):
        return (f"ClaimPlain({self.claim_id!r}, {self.policy_type!r}, "
                f"{self.reserve}, {self.paid})")


# ---- Part 2: Dataclass (decorator writes __init__, __repr__, __eq__) ----
@dataclass
class Claim:
    claim_id: str
    policy_type: str
    reserve: float
    paid: float = 0.0          # typed field with a default value

    def reserve_burn(self) -> float:
        return round(self.paid / self.reserve * 100, 1)


# ---- Standalone function: same math, different call style ----
def reserve_burn_fn(claim) -> float:
    return round(claim.paid / claim.reserve * 100, 1)


# Demo
c1 = ClaimPlain("CLM-5001", "auto", 5000.0, 1200.0)
print("Plain class:", c1)
print("Plain burn:", c1.reserve_burn())

c2 = Claim("CLM-5001", "auto", 5000.0, 1200.0)
print("\nDataclass:", c2)
print("Dataclass burn:", c2.reserve_burn())

# __eq__ is free: compares field by field
c3 = Claim("CLM-5001", "auto", 5000.0, 1200.0)
print("c2 == c3:", c2 == c3)

# Default field: paid is optional at construction time
c4 = Claim("CLM-5002", "property", 12000.0)
print("With default paid:", c4)
print("Burn with 0 paid:", c4.reserve_burn())

# Method vs. function: same result
print("\nMethod call:", c2.reserve_burn())
print("Function call:", reserve_burn_fn(c2))

**Output explained**

- `ClaimPlain.__repr__` is written by hand and controls how the object prints. Without it, you would see `<__main__.ClaimPlain object at 0x...>`.
- `@dataclass` generates `__init__` and `__repr__` automatically from the type-annotated fields. The output is clean and readable.
- `c2 == c3` is `True` because `@dataclass` generates `__eq__` that compares field values, not object identity (memory addresses). Plain classes default to identity comparison.
- `c4 = Claim("CLM-5002", "property", 12000.0)` omits `paid`; it defaults to `0.0`. Calling `c4.reserve_burn()` with 0 paid returns 0.0%.

**When to use each**:
- Prefer `@dataclass` for data containers with no complex initialization logic.
- Use a plain class when you need `__post_init__` validation, class variables, or complex inheritance.

---
## Advanced Track: A4 - itertools Pipelines

### What it is
The `itertools` standard library module provides memory-efficient tools for working with iterators. The key insight: itertools functions return **iterators**, not lists. They produce values one at a time, so they can handle arbitrarily large sequences without loading everything into memory.

| Function | What it does |
|----------|-------------|
| `chain(*iterables)` | treats multiple iterables as one stream |
| `islice(iterable, n)` | take first n items without materializing the rest |
| `accumulate(iterable)` | running total (or any binary operation) |
| `groupby(iterable, key)` | group consecutive items by key (sort first!) |

### Why data engineers care
Large datasets cannot always fit in memory. Iterator pipelines process one record at a time. PySpark's transformation graph and pandas chaining are the same idea at scale.

In [ ]:
from itertools import chain, islice, accumulate, groupby

batch1 = [{"claim_id": "CLM-1", "paid": 1200.0, "policy_type": "auto"},
          {"claim_id": "CLM-2", "paid": 5000.0, "policy_type": "property"}]
batch2 = [{"claim_id": "CLM-3", "paid":  800.0, "policy_type": "auto"},
          {"claim_id": "CLM-4", "paid": 4000.0, "policy_type": "liability"}]

# chain: merge two batches into one stream without building a new list
all_claims = list(chain(batch1, batch2))
print("Chained ids:", [c["claim_id"] for c in all_claims])

# islice: take the first 2 without iterating the entire stream
first_two = list(islice(chain(batch1, batch2), 2))
print("First two:", [c["claim_id"] for c in first_two])

# accumulate: running total of paid amounts
paids = [c["paid"] for c in all_claims]
running_total = list(accumulate(paids))
print("Running total:", running_total)

# groupby: group by policy_type - MUST sort by the same key first
all_claims.sort(key=lambda c: c["policy_type"])
print("\nGrouped by policy type:")
for ptype, group in groupby(all_claims, key=lambda c: c["policy_type"]):
    ids = [c["claim_id"] for c in group]  # consume the group iterator immediately
    print(f"  {ptype}: {ids}")

**Output explained**

- `chain(batch1, batch2)` produces all four claims without copying either list. Wrapping in `list()` materializes the iterator only when needed.
- `islice(chain(...), 2)` takes exactly 2 items from the chained stream. Crucially, the rest of `batch2` is never touched. This is useful when peeking at a large dataset.
- `accumulate([1200, 5000, 800, 4000])` produces `[1200, 6200, 7000, 11000]`: each value is added to the running total from the previous step.

**`groupby` warning**: `groupby` groups CONSECUTIVE items that share the same key. If your data is not sorted by the grouping key, groups will be fragmented. Always sort first. (This is different from SQL `GROUP BY`, which groups all matching rows regardless of order.)

Also notice: `for ptype, group in groupby(...)` gives you `group` as a lazy iterator. You MUST consume it (e.g., `list(group)`) inside the loop. If you call `groupby` again before consuming, the previous group becomes empty.

---
## Advanced Track: A5 - Decorators

### What it is
A **decorator** is a function that wraps another function to add behavior without modifying the original function's code. The `@name` syntax applies a decorator.

```python
def my_decorator(func):
    def wrapper(*args, **kwargs):   # accepts any arguments
        # do something before
        result = func(*args, **kwargs)
        # do something after
        return result
    return wrapper

@my_decorator
def my_function(): ...
```

`@functools.wraps(func)` copies the original function's name, docstring, and other metadata to the wrapper. Without it, `my_function.__name__` would be `"wrapper"`.

### Why data engineers care
Decorators power logging, timing, retry logic, caching, and authentication in real pipelines. Libraries like `tenacity` (retries), `functools.lru_cache` (caching), and AWS Lambda all use decorators as their primary interface.

In [ ]:
import time
from functools import wraps


def timer(func):
    """A decorator that prints how long a function took to run."""
    @wraps(func)                        # preserve func's name and docstring
    def wrapper(*args, **kwargs):       # accept any positional and keyword args
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"  {func.__name__!r} took {elapsed:.4f}s")
        return result
    return wrapper


@timer
def total_paid(claims):
    """Sum the paid amount across a list of claims."""
    return sum(c["paid"] for c in claims)


@timer
def find_breaches(claims):
    """Return claims where paid exceeds reserve."""
    return [c for c in claims if c["paid"] > c["reserve"]]


# Generate a large batch of synthetic claims
claims = [{"claim_id": f"CLM-{i}", "reserve": float(i * 100), "paid": float(i * 95)}
          for i in range(1, 10001)]

result = total_paid(claims)
print(f"  total paid: ${result:,.2f}")

breaches = find_breaches(claims)
print(f"  {len(breaches)} breaches found")

# The decorator does not change the function's identity
print(f"\nFunction name preserved: {total_paid.__name__}")
print(f"Docstring preserved: {total_paid.__doc__}")

**Output explained**

Each decorated function prints its own timing line before the caller sees the result. The timing is added transparently: `total_paid(claims)` still returns the sum, and the caller does not need to know the wrapper exists.

- `@wraps(func)` is essential. Without it, `total_paid.__name__` would be `"wrapper"`, which confuses debugging, logging, and documentation tools.
- `*args, **kwargs` lets the wrapper forward any argument signature to the original function. The wrapper does not need to know what arguments `func` accepts.

**Real-world extensions**: add a `logger.info(...)` call inside `wrapper` to produce a structured log entry for every function call. Add a `try/except` around `func(...)` to catch and re-raise with richer context. Chain multiple decorators: `@timer @log @retry`.

---
## Mini Project 1: Claims Intake Pipeline

### What this brings together
This is the real `Activity_1_Mini_Capstone_Claims_Intake/` mini project: CSV reading, JSON reading, data cleaning and validation, dictionary aggregation, and file writing, chained into one realistic end-to-end pipeline. A data engineer rarely works on isolated drills; this is what the job looks like.

### The scenario
Charter Oak Mutual's claim export is messy: money fields have dollar signs and commas, some are blank or say `N/A`, a few rows are missing a claim id, and one claim id is duplicated. A separate JSON feed carries each claim's payment events.

Your pipeline must:
1. **Core**: read the raw CSV, clean the money fields, and reject invalid or duplicate rows.
2. **Challenge**: aggregate the clean claims by policy type and compute a loss ratio.
3. **Stretch**: roll up the nested payment events per claim and flag any claim whose payments exceed its reserve, for the Special Investigations Unit (SIU).


In [ ]:
import csv, json
from pathlib import Path

# This loads the REAL Mini Project 1 data and reproduces its real solution logic
# and real numbers. Paths are relative to this notebook's own folder (Day 2 root).
RAW_CSV = Path("Activity_1_Mini_Capstone_Claims_Intake/data/claims_raw.csv")
PAYMENTS_JSON = Path("Activity_1_Mini_Capstone_Claims_Intake/data/claim_payments.json")


def parse_money(raw):
    if raw is None:
        return None
    cleaned = raw.strip().replace("$", "").replace(",", "")
    if cleaned == "" or cleaned.upper() == "N/A":
        return None
    try:
        value = float(cleaned)
    except ValueError:
        return None
    return None if value < 0 else round(value, 2)


# ---- CORE: clean and validate ----
clean = []
rejected = 0
seen = set()
with open(RAW_CSV, newline="") as f:
    for row in csv.DictReader(f):
        claim_id = (row.get("claim_id") or "").strip()
        reserve = parse_money(row.get("reserve"))
        paid = parse_money(row.get("paid"))
        if not claim_id or reserve is None or paid is None or claim_id in seen:
            rejected += 1
            continue
        seen.add(claim_id)
        clean.append({
            "claim_id": claim_id,
            "policy_type": (row.get("policy_type") or "").strip().lower() or "unknown",
            "reserve": reserve,
            "paid": paid,
        })

print("=== CORE: Intake summary (real Mini Project 1 data) ===")
print(f"Valid claims:  {len(clean)}")
print(f"Rejected rows: {rejected}")
print(f"Total reserve: ${sum(c['reserve'] for c in clean):,.2f}")
print(f"Total paid:    ${sum(c['paid'] for c in clean):,.2f}")

# ---- CHALLENGE: aggregate by policy type ----
summary = {}
for c in clean:
    bucket = summary.setdefault(c["policy_type"], {"count": 0, "reserve": 0.0, "paid": 0.0})
    bucket["count"] += 1
    bucket["reserve"] += c["reserve"]
    bucket["paid"] += c["paid"]

print("\n=== CHALLENGE: Loss ratio by policy type ===")
for ptype in sorted(summary):
    b = summary[ptype]
    lr = round(b["paid"] / b["reserve"] * 100, 2) if b["reserve"] else 0.0
    print(f"  {ptype:<10} count={b['count']}  reserve=${b['reserve']:,.2f}  "
          f"paid=${b['paid']:,.2f}  loss_ratio={lr}%")

# ---- STRETCH: nested payment rollup and SIU review ----
with open(PAYMENTS_JSON) as f:
    payments = json.load(f)
reserve_by_id = {c["claim_id"]: c["reserve"] for c in clean}

print("\n=== STRETCH: SIU reserve-breach review ===")
for claim_id, events in payments.items():
    total_paid = round(sum(e["amount"] for e in events), 2)
    reserve = reserve_by_id.get(claim_id)
    if reserve is not None and total_paid > reserve:
        print(f"  {claim_id}: paid ${total_paid:,.2f} vs reserve ${reserve:,.2f} "
              f"(overage ${round(total_paid - reserve, 2):,.2f})")

**What this pipeline demonstrates**

This is the real `Activity_1_Mini_Capstone_Claims_Intake/` data, loaded and processed
with the same clean-validate-aggregate-flag logic as the reference solution. If you
completed Mini Project 1, these numbers should match your own output exactly: 16
valid claims, 6 rejected, $144,000.00 total reserve, $87,900.50 total paid, and two
SIU breaches (CLM-1009, CLM-1016).

1. **Two-source ingestion**: CSV (structured, flat) and JSON (structured, nested)
   are loaded separately and joined on a shared key (`claim_id`). This is the most
   common pattern in real data engineering.
2. **Key-based join**: loading claims into a dict keyed by `claim_id` makes the
   join O(1) per payment record. In Week 3 you will do the same thing in BigQuery
   with a `JOIN` clause.
3. **Reject, don't crash**: rows with a missing id, missing or invalid money
   fields, or a duplicate id are rejected and counted, not allowed to crash the
   pipeline or silently corrupt the totals.
4. **Derived fields**: `loss_ratio` and the SIU overage are computed after loading,
   the "transform" step in ETL.
5. **Flagging**: claims that breach reserve are surfaced for the Special
   Investigations Unit, the same way a real pipeline would route an exception to a
   supervisor queue instead of just logging it.

---
## Mini Project 2: Appointment No-Show Analysis

### The scenario
A health plan wants to reduce appointment no-shows. You have real appointment records with patient attributes (age, neighborhood, scholarship status, hypertension, diabetes), whether an SMS reminder was sent, and whether the patient showed up. The job: quantify the no-show problem, then interpret it for the business, including a genuinely counterintuitive result about SMS reminders.

### What this brings together
`csv` module, aggregation and rate calculations, grouping by category, and turning numbers into a business recommendation. This is the "half a data engineer is reading the numbers" half of the day.


In [ ]:
import csv
from pathlib import Path

# This loads the REAL Mini Project 2 data and reproduces its real solution logic
# and real numbers. Path is relative to this notebook's own folder (Day 2 root).
DATA = Path("Activity_2_No_Show_Analysis/data/no_shows.csv")


def is_yes(value):
    return value.strip().lower() in ("yes", "y")


def is_no_show(row):
    return row["No-show"].strip().lower() in ("yes", "y")


def rate(rows):
    """Return (appointments, no_shows, percent) for a list of rows."""
    if not rows:
        return (0, 0, 0.0)
    no = sum(1 for r in rows if is_no_show(r))
    return (len(rows), no, round(no / len(rows) * 100, 1))


def age_group(age):
    if age < 35:
        return "18-34"
    if age < 55:
        return "35-54"
    return "55+"


def show(label, rows):
    appts, no, pct = rate(rows)
    print(f"  {label:<16} {appts:>6} appts, {no:>5} no-shows, {pct:>5}%")


with open(DATA, newline="") as f:
    rows = list(csv.DictReader(f))

print(f"=== Mini Project 2: {len(rows):,} real appointment records ===")
appts, no, pct = rate(rows)
print(f"Overall no-show rate: {pct}%\n")

print("By SMS reminder:")
show("SMS received", [r for r in rows if is_yes(r["SMS_received"])])
show("no SMS", [r for r in rows if not is_yes(r["SMS_received"])])

print("\nBy age group:")
for g in ("18-34", "35-54", "55+"):
    show(g, [r for r in rows if age_group(int(r["Age"])) == g])

print("\nBy condition:")
show("hypertension", [r for r in rows if is_yes(r["Hypertension"])])
show("diabetes", [r for r in rows if is_yes(r["Diabetes"])])

**What this analysis demonstrates**

This is the real `Activity_2_No_Show_Analysis/` dataset (12,573 real appointment
records), loaded and broken down with the same logic as the reference solution. If
you completed Mini Project 2, these numbers should match your own output exactly:
overall 20.1%, SMS received 27.5% versus no SMS 16.4%.

1. **The SMS paradox**: appointments with an SMS reminder show a HIGHER no-show
   rate (27.5%) than those without (16.4%). This is the headline result. Reminders
   are usually sent for appointments booked far in advance, and appointments booked
   far ahead are more likely to be forgotten or become inconvenient. The SMS is a
   marker of "booked long ago," not the cause of the no-show. This is a confounding
   variable, and spotting it is the real skill.
2. **Flat factors**: age group and health condition move the no-show rate only a
   little compared to the SMS split. Numbers that barely move tell you where not
   to spend effort.
3. **A data engineer's job is not done at the print statement.** Producing the rate
   is half the work. The other half is the framing above: what the numbers can and
   cannot prove, and what data (like lead time between booking and appointment)
   you would ask for next to be sure.

---
## What Comes Next

You have now covered the full Python toolkit needed for a data engineer:

| Skill | Where you will use it again |
|-------|-----------------------------|
| Variables, types, f-strings | Everywhere |
| Conditionals | Routing logic in every pipeline |
| Loops | Batch processing before pandas / PySpark |
| Lists and dicts | Every data structure in Python |
| Functions | Every reusable pipeline stage |
| Nested data | JSON API responses (Day 3) |
| Classes | SDK clients, pipeline stages |
| File I/O | ETL source and sink |
| CSV / JSON I/O | Source ingestion (Days 3-4) |
| Lambdas, map, filter | PySpark RDD transformations (Week 5) |
| SQLite | SQL practice before BigQuery (Week 3) |
| Error handling | Resilient API ingestion (Day 3) |
| collections | Group-and-count before pandas groupby (Day 4) |
| datetime | Timestamp parsing everywhere |
| Comprehensions | Idiomatic Python data transformations |
| Pattern matching | Routing complex API responses |
| Dataclasses | Typed pipeline models |
| itertools | Memory-efficient large-data pipelines |
| Decorators | Logging, timing, retry in production pipelines |

**Day 3 preview**: you will use `requests` and `httpx` to call a live REST API and process its JSON response with the skills from this notebook.